# Tool 5 — LLM Agent Orchestrator

An LLM (Claude via API) orchestrates Tools 1–4 in whatever order it judges useful,
then produces a structured diagnostic impression.

**Requires:** `oct_best.pth` and `retrieval_encoder.pth` checkpoints.

**Agent loop:**
```
Image
  → Tool 1: coarse classify         (OCTNet → label + confidence)
  → Tool 2: localize                (GradCAM++ → heatmap + ROI description)
  → Tool 3: zoom-and-reanalyze      (crop ROI → second prediction)
  → Tool 4: retrieve knowledge      (embed query → top-k reference snippets)
  → LLM synthesizes all results
  → Structured output: {finding, confidence, localization, justification}
```

The agent decides *which* tools to call and *in what order*, and may call some multiple
times (e.g. retrieve once per finding candidate, or skip zoom if confidence is high).

## 0. Setup
Paste imports, config, and model definitions from your main notebook.
Then load both checkpoints.

In [ ]:
# ── Paste from main notebook ──────────────────────────────────────────────────
# Imports, config (DEVICE, CLASS_NAMES, IMG_SIZE, CKPT_PATH)
# OCTNet + GradCAMpp class definitions
# val_tf transform

# ── Paste from tool4 notebook ─────────────────────────────────────────────────
# WordTokenizer, SmallEncoder, CORPUS, retrieve()

# ── Paste from tool3 notebook ─────────────────────────────────────────────────
# zoom_and_reanalyze()

# ── Load OCTNet ───────────────────────────────────────────────────────────────
eval_model = OCTNet(NUM_CLASSES).to(DEVICE)
eval_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
eval_model.eval()
cam_fn = GradCAMpp(eval_model, eval_model.cam_layer)

# ── Load retrieval encoder ────────────────────────────────────────────────────
CKPT_EMB = 'retrieval_encoder.pth'
encoder.load_state_dict(torch.load(CKPT_EMB, map_location=DEVICE))
encoder.eval()

# Pre-embed corpus
corpus_texts  = [text  for _, text  in CORPUS]
corpus_labels = [label for label, _ in CORPUS]
corpus_embs   = embed_texts(corpus_texts)

print('All models loaded.')

## 1. Tool Definitions (Python callables)
Wrap each capability as a clean function the agent can call by name.

In [ ]:
import json
from PIL import Image
import torch
import torch.nn.functional as F
from torchvision import transforms


def tool_coarse_classify(image_path: str) -> dict:
    """
    Tool 1: Run OCTNet on the full image.
    Returns predicted class, confidence, and all class probabilities.
    """
    infer_tf = transforms.Compose([
        transforms.Grayscale(3),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3),
    ])
    pil = Image.open(image_path)
    img_t = infer_tf(pil)
    cam, cls_idx, probs = cam_fn(img_t)
    return {
        'predicted_class': CLASS_NAMES[cls_idx],
        'confidence':      round(float(probs[cls_idx]), 4),
        'all_probs':       {c: round(float(p), 4) for c, p in zip(CLASS_NAMES, probs)},
    }


def tool_localize(image_path: str) -> dict:
    """
    Tool 2: Run GradCAM++ and describe the activated region.
    Returns the dominant spatial region of activation (quadrant description).
    """
    infer_tf = transforms.Compose([
        transforms.Grayscale(3),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3),
    ])
    pil = Image.open(image_path)
    img_t = infer_tf(pil)
    cam, cls_idx, probs = cam_fn(img_t)

    # Describe location in anatomically meaningful terms for OCT
    h, w = cam.shape
    peak_y, peak_x = divmod(int(cam.argmax()), w)
    frac_x, frac_y = peak_x / w, peak_y / h

    horiz = 'nasal' if frac_x < 0.4 else ('temporal' if frac_x > 0.6 else 'central')
    vert  = 'superior' if frac_y < 0.4 else ('inferior' if frac_y > 0.6 else 'mid')
    region = f'{vert}-{horiz}' if horiz != 'central' or vert != 'mid' else 'central foveal'

    activation_spread = float((cam > 0.4).sum()) / cam.size  # fraction of image activated

    return {
        'predicted_class':    CLASS_NAMES[cls_idx],
        'peak_activation_at': region,
        'activation_spread':  round(activation_spread, 3),
        'spread_description': 'focal' if activation_spread < 0.15 else ('moderate' if activation_spread < 0.35 else 'diffuse'),
    }


def tool_zoom_reanalyze(image_path: str) -> dict:
    """
    Tool 3: Crop the GradCAM-activated region and re-classify at full resolution.
    Returns both the original and crop predictions, and whether they agree.
    """
    pil = Image.open(image_path)
    result = zoom_and_reanalyze(pil, eval_model, cam_fn, CLASS_NAMES)
    return {
        'full_image_pred':  result['original_pred'],
        'full_image_conf':  round(result['original_conf'], 4),
        'crop_pred':        result['crop_pred'],
        'crop_conf':        round(result['crop_conf'], 4),
        'agreement':        result['agreement'],
        'bbox':             result['bbox'],
    }


def tool_retrieve_knowledge(query: str, top_k: int = 3) -> dict:
    """
    Tool 4: Retrieve reference radiology text relevant to a query.
    Args:
        query: E.g. 'CNV finding' or 'intraretinal fluid cysts diabetic'.
        top_k: Number of snippets to return.
    Returns the top-k most relevant reference snippets and their scores.
    """
    results = retrieve(query, top_k=top_k)
    return {
        'query':   query,
        'results': [{'rank': r['rank'], 'label': r['label'],
                     'text': r['text'], 'score': round(r['score'], 4)} for r in results],
    }


# Registry — maps tool names to callables
TOOL_REGISTRY = {
    'coarse_classify':   tool_coarse_classify,
    'localize':          tool_localize,
    'zoom_reanalyze':    tool_zoom_reanalyze,
    'retrieve_knowledge': tool_retrieve_knowledge,
}

print('Tool registry ready:', list(TOOL_REGISTRY.keys()))

## 2. Agent Prompts

In [ ]:
SYSTEM_PROMPT = """\
You are a diagnostic AI assistant for OCT (optical coherence tomography) retinal imaging.
You have access to four tools that you can call to analyze an image:

1. coarse_classify(image_path) → {predicted_class, confidence, all_probs}
   Run a CNN classifier on the full image to get an initial prediction.
   Classes: CNV, DME, DRUSEN, NORMAL.

2. localize(image_path) → {predicted_class, peak_activation_at, spread_description}
   Run GradCAM++ to identify which region of the image most influenced the prediction.
   Returns the anatomical region (e.g. 'central foveal', 'superior-nasal') and whether
   the activation is focal or diffuse.

3. zoom_reanalyze(image_path) → {full_image_pred, crop_pred, agreement, crop_conf}
   Crop the most activated region and re-run the classifier on it.
   Useful when initial confidence is moderate (<85%) or when you want to verify a finding.

4. retrieve_knowledge(query, top_k=3) → {results: [{label, text, score}]}
   Retrieve relevant radiology reference text for a finding or symptom description.
   Use this to ground your explanation in clinical knowledge.
   Example queries: 'CNV finding', 'subretinal fluid appearance', 'drusen AMD'.

INSTRUCTIONS:
- Always start with coarse_classify and localize.
- Call zoom_reanalyze if initial confidence is below 85% OR if the finding is ambiguous.
- Call retrieve_knowledge for the predicted class (and for runner-up classes if probabilities
  are close, i.e. within 15% of each other).
- You may call tools multiple times or in any order.
- After gathering evidence, produce a structured JSON output with these exact keys:
  {
    "finding": "<CNV|DME|DRUSEN|NORMAL>",
    "confidence": <0.0-1.0>,
    "localization": "<description of where in the image the finding is located>",
    "supporting_evidence": ["<list of clinical features from retrieved knowledge that match>"],
    "justification": "<2-3 sentence natural language explanation referencing each tool used>",
    "uncertainty_flags": ["<any concerns: low confidence, disagreement between passes, etc>"]
  }

Call tools by responding ONLY with:
TOOL: <tool_name>
ARGS: <JSON args>

When you have enough evidence, respond with:
FINAL: <JSON output>
"""

print('System prompt defined.')

## 3. Agent Loop

In [ ]:
import anthropic
import re

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from environment


def parse_tool_call(text: str) -> tuple[str, dict] | None:
    """Parse TOOL: name / ARGS: {...} from model output."""
    tool_m = re.search(r'TOOL:\s*(\w+)', text)
    args_m = re.search(r'ARGS:\s*(\{.*?\})', text, re.DOTALL)
    if not tool_m:
        return None
    tool_name = tool_m.group(1).strip()
    args = json.loads(args_m.group(1)) if args_m else {}
    return tool_name, args


def parse_final(text: str) -> dict | None:
    """Parse FINAL: {...} from model output."""
    m = re.search(r'FINAL:\s*(\{.*\})', text, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(1))
    except json.JSONDecodeError:
        return None


def run_agent(image_path: str, verbose: bool = True) -> dict:
    """
    Run the full agentic diagnostic pipeline on an image.

    Args:
        image_path: Path to the OCT image file.
        verbose:    Print each tool call and result.

    Returns:
        Structured diagnostic dict with keys:
        finding, confidence, localization, supporting_evidence, justification, uncertainty_flags.
    """
    messages = [
        {
            'role': 'user',
            'content': f'Please analyze this OCT image and provide a structured diagnostic impression.\nImage path: {image_path}'
        }
    ]

    tool_calls_log = []
    max_turns = 12  # safety limit

    for turn in range(max_turns):
        response = client.messages.create(
            model='claude-sonnet-4-6',
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            messages=messages,
        )
        assistant_text = response.content[0].text
        messages.append({'role': 'assistant', 'content': assistant_text})

        # ── Check if agent is done ─────────────────────────────────────────────
        final = parse_final(assistant_text)
        if final:
            final['_tool_calls'] = tool_calls_log
            if verbose:
                print(f'\n=== FINAL OUTPUT (after {turn+1} turns) ===')
                print(json.dumps(final, indent=2))
            return final

        # ── Parse and execute tool call ────────────────────────────────────────
        parsed = parse_tool_call(assistant_text)
        if not parsed:
            # Agent didn't call a tool or produce a final — shouldn't happen, but guard
            break

        tool_name, args = parsed
        if tool_name not in TOOL_REGISTRY:
            tool_result = {'error': f'Unknown tool: {tool_name}'}
        else:
            try:
                tool_result = TOOL_REGISTRY[tool_name](**args)
            except Exception as e:
                tool_result = {'error': str(e)}

        tool_calls_log.append({'tool': tool_name, 'args': args, 'result': tool_result})

        if verbose:
            print(f'Turn {turn+1} | {tool_name}({args}) → {json.dumps(tool_result)[:120]}...')

        # Feed result back to agent
        messages.append({
            'role': 'user',
            'content': f'TOOL_RESULT for {tool_name}:\n{json.dumps(tool_result, indent=2)}'
        })

    return {'error': 'Agent did not produce a final output within max_turns', '_tool_calls': tool_calls_log}


print('run_agent() defined.')

## 4. Run on a single example

In [ ]:
# Pick a test image — replace with any image path
sample_path, sample_label = test_ds.samples[0]
true_class = CLASS_NAMES[sample_label]
print(f'Image: {sample_path}')
print(f'True label: {true_class}\n')

result = run_agent(sample_path, verbose=True)

print(f'\n→ Predicted: {result.get("finding")}  |  True: {true_class}')
print(f'→ Correct: {result.get("finding") == true_class}')

## 5. Ablation Study
Remove each tool in turn and measure the impact on accuracy.
This is the core empirical contribution from your proposal.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

# For ablation we use a stripped-down non-agentic version:
# disable each tool by making it return an empty/neutral result.

def ablation_run(image_path: str, disabled_tools: list[str]) -> str:
    """
    Run the pipeline with certain tools disabled.
    Disabled tools return a placeholder so the agent knows they're unavailable.
    Returns the predicted class name.
    """
    # Temporarily patch the registry
    originals = {}
    for t in disabled_tools:
        originals[t] = TOOL_REGISTRY[t]
        TOOL_REGISTRY[t] = lambda **_: {'error': 'tool disabled for ablation'}

    result = run_agent(image_path, verbose=False)

    # Restore registry
    for t, fn in originals.items():
        TOOL_REGISTRY[t] = fn

    return result.get('finding', 'UNKNOWN')


# Run ablation on a small evaluation subset (first 40 test images per class)
subset = []
per_class = {c: 0 for c in CLASS_NAMES}
for path, label in test_ds.samples:
    cname = CLASS_NAMES[label]
    if per_class[cname] < 40:
        subset.append((path, label))
        per_class[cname] += 1
print(f'Ablation subset: {len(subset)} images')

ABLATION_CONDITIONS = {
    'Full pipeline':            [],
    'No zoom (−Tool3)':         ['zoom_reanalyze'],
    'No retrieval (−Tool4)':    ['retrieve_knowledge'],
    'No localization (−Tool2)': ['localize'],
    'Classifier only (−2,3,4)': ['localize', 'zoom_reanalyze', 'retrieve_knowledge'],
}

ablation_results = {}
for condition, disabled in ABLATION_CONDITIONS.items():
    preds, trues = [], []
    for path, label in tqdm(subset, desc=condition):
        pred = ablation_run(path, disabled)
        preds.append(pred)
        trues.append(CLASS_NAMES[label])
    acc = accuracy_score(trues, preds)
    f1  = f1_score(trues, preds, average='macro', labels=CLASS_NAMES, zero_division=0)
    ablation_results[condition] = {'accuracy': acc, 'macro_f1': f1}
    print(f'{condition:35s} | Acc {acc*100:.1f}% | F1 {f1*100:.1f}%')

## 6. Print Ablation Table

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {'Condition': cond, 'Accuracy (%)': f"{v['accuracy']*100:.1f}", 'Macro F1 (%)': f"{v['macro_f1']*100:.1f}"}
    for cond, v in ablation_results.items()
])
print(df.to_string(index=False))

## 7. Visualize one full agent trace

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from PIL import Image as PILImage


def visualize_agent_trace(image_path: str, result: dict):
    """Show the image, GradCAM heatmap, and a summary of the agent's reasoning."""
    tool_calls = result.get('_tool_calls', [])

    # Get GradCAM from localize result if available
    infer_tf = transforms.Compose([
        transforms.Grayscale(3), transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(), transforms.Normalize([0.5]*3, [0.5]*3),
    ])
    pil = PILImage.open(image_path)
    img_t = infer_tf(pil)
    cam, _, _ = cam_fn(img_t)
    img_np = img_t.numpy().transpose(1, 2, 0) * 0.5 + 0.5
    img_gray = img_np.mean(2)
    cam_up = np.array(PILImage.fromarray((cam * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE)))/255.
    overlay = (0.55 * np.stack([img_gray]*3, 2) + 0.45 * plt.cm.jet(cam_up)[:,:,:3]).clip(0,1)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(
        f"Agent Diagnosis: {result.get('finding', '?')}  "
        f"(conf={result.get('confidence', 0)*100:.0f}%)  "
        f"| {len(tool_calls)} tool calls",
        fontsize=13
    )

    axes[0].imshow(img_gray, cmap='gray'); axes[0].set_title('Input OCT'); axes[0].axis('off')
    axes[1].imshow(overlay);               axes[1].set_title('GradCAM++ (localization)'); axes[1].axis('off')

    # Reasoning summary text panel
    axes[2].axis('off')
    summary = [
        f"Finding: {result.get('finding', '?')}",
        f"Confidence: {result.get('confidence', 0)*100:.0f}%",
        f"Location: {result.get('localization', 'N/A')}",
        "",
        "Justification:",
    ] + [f"  {line}" for line in (result.get('justification', '') or '').split('. ') if line] + [
        "",
        "Supporting evidence:",
    ] + [f"  • {e}" for e in (result.get('supporting_evidence') or [])[:3]] + [
        "",
    ] + ([f"⚠ {f}" for f in (result.get('uncertainty_flags') or [])] or ['✓ No flags'])

    axes[2].text(0.02, 0.98, '\n'.join(summary), transform=axes[2].transAxes,
                 fontsize=8, va='top', family='monospace',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout()
    plt.savefig('tool5_agent_trace.png', dpi=150, bbox_inches='tight')
    plt.show()


# Visualize the result from Section 4
visualize_agent_trace(sample_path, result)